# Brownian lineage steps versus per-release steps, on the 98-benchmark data
The Brownian fits win on LOO — **+99.7 ± 29.2** nats for the simplified BM + estimated noise gap — but they do **not** remove the two basins: all three fits split the same way, on whether GBAEval loads on easy knowledge or on the fluid axis.
The rate hierarchy is empty (36 vendor rates inside **9–20%** of one vendor's own CI) and the noise gap fires on 3 saturated benchmarks, not on the new ones.
**Data caveat:** the no-BM fit was sampled on **4,447** observations, not 4,445 — the 2 GBAEval harness rows were dropped after it ran, so its log-likelihood is subset to the common 4,445 before any LOO.

In [1]:
import sys, json, itertools
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "fit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import xarray as xr
import arviz as az
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import expit
from scipy.stats import beta as beta_dist

from config import ECI_EPS
from data import load_eci_data, load_benchmark_floors, _known_release_date_by_model
from analysis import mirt_identified_rhat

# fit B was sampled in a scratch worktree and never moved into results/
B_DIR = Path("/private/tmp/claude-501/-Users-yassineessifi-Desktop-ECI-Bayesian"
             "/1dac9d5d-2e80-433e-8bab-4cbdef0fc499/scratchpad/bm_ratehier2/results"
             "/mirt_humanprior_lineageprior_lineagebm_floors_ceilings")
PATHS = {
    "A": ROOT / "results/mirt_humanprior_lineageprior_floors"
                "/trace_mirt_k3_humanprior_lineageprior_floors.nc",
    "B": B_DIR / "trace_mirt_k3_humanprior_lineageprior_lineagebm_floors_ceilings.nc",
    "C": ROOT / "results/mirt_humanprior_lineageprior_lineagebm_floors_ceilings_ceilnoise"
                "/trace_mirt_k3_humanprior_lineageprior_lineagebm_floors_ceilings_ceilnoise.nc",
}
LABEL = {"A": "A · no-BM, 3PL floors (12×3000)",
         "B": "B · BM + rate hierarchy + fixed 4PL (8×2000)",
         "C": "C · BM simplified + fixed 4PL + est. noise gap (6×2000)"}
SHORT = {"A": "A · no-BM", "B": "B · BM + rate hier.", "C": "C · BM + noise gap"}
PLOTS = ROOT / "plots/k3_bm_vs_nobm"
PLOTS.mkdir(parents=True, exist_ok=True)
THIN = 5

C = dict(blue="#0072B2", sky="#56B4E9", orange="#E69F00", verm="#D55E00",
         green="#009E73", pink="#CC79A7", gray="#999999", dark="#333333")
FCOL = {"A": C["gray"], "B": C["orange"], "C": C["blue"]}

def show(fig, name):
    fig.update_layout(template="plotly_white", title=dict(x=0.5, font=dict(size=13)))
    fig.write_image(PLOTS / f"{name}.png", scale=2)
    fig.show()

# ── data, shared by every claim ──────────────────────────────────────────────
data = load_eci_data(include_all_benchmarks=True)
BEN = data.blookup.sort_values("benchmark_idx")["benchmark"].tolist()
MOD = data.mlookup.sort_values("model_idx")["model"].tolist()
FLOOR = load_benchmark_floors(data)
YOBS = np.clip(data.scores, ECI_EPS, 1.0 - ECI_EPS)
RAW = pd.read_csv(ROOT / "data/processed/benchmarks_merged.csv")
RAW["release_date"] = pd.to_datetime(RAW["release_date"], errors="coerce")

# the 13 benchmarks the 2026-07-27/28 refresh added (85 -> 98)
NEW_IN = ["EBR-bench", "GBAEval", "GDPval", "Surface Evolver Bench"]      # in Epoch's ECI
NEW_OUT = ["AlgoTune", "BlueprintBench 2", "DeepSWE", "FrontierMath v1",
           "FrontierMath Tier 4 v1", "GDP.pdf", "MindCube", "ProofBench",
           "SpatialViz-Bench"]                                           # not in it
NEW = NEW_IN + NEW_OUT
OLD = [b for b in BEN if b not in NEW]
assert set(NEW) <= set(BEN) and len(NEW) == 13 and len(OLD) == 85
G_IN, G_OUT, G_OLD = "new / in Epoch ECI", "new / not in ECI", "pre-existing"
GCOL = {G_IN: C["verm"], G_OUT: C["orange"], G_OLD: C["gray"]}

# the 7 classic commonsense items — the ONLY thing that names the easy axis
CLASSIC = ["GSM8K", "OpenBookQA", "ARC (AI2)", "PIQA", "HellaSwag",
           "Adversarial NLI", "CSQA2"]
PROBE = {"Easy knowledge": CLASSIC,
         "Hard math + science": ["FrontierMath Tier 4", "OTIS Mock AIME 2024-2025",
                                 "MATH Level 5", "FrontierMath"],
         "Fluid / abstract": ["VPCT", "ARC-AGI-2", "ARC-AGI"]}
PIDX = {k: [BEN.index(b) for b in v] for k, v in PROBE.items()}

# the 2 rows dropped from the data AFTER fit A was sampled; positions in A's
# observation vector are asserted below against A's own log-likelihood
DROP_A = [29, 30]
DROP_MODELS = ["glm-5.1", "MiniMax-M2.7"]

PERMS = list(itertools.permutations(range(3)))
FITS = {}

for tag, path in PATHS.items():
    want = ["A", "theta", "D", "sigma_b"]
    extra = {"B": ["lin_rate"], "C": ["ceiling_gap", "ceiling_d"]}.get(tag, [])
    post = xr.open_dataset(path, group="posterior")[want + extra] \
             .isel(draw=slice(None, None, THIN)).load()
    stats = xr.open_dataset(path, group="sample_stats")[["logp", "diverging"]].load()
    NC = post.sizes["chain"]
    Amed = np.median(post["A"].values, axis=1)          # (chain, bench, 3)
    THmed = np.median(post["theta"].values, axis=1)     # (chain, model, 3)

    def match(c1, c2):
        """Best axis permutation of chain c2 onto c1, with matched per-axis corrs."""
        M = np.corrcoef(Amed[c1].T, Amed[c2].T)[:3, 3:]
        p = max(PERMS, key=lambda q: sum(M[i, q[i]] for i in range(3)))
        return list(p), np.array([M[i, p[i]] for i in range(3)])

    # alignment score = WORST matched axis; within-basin >0.91, across <0.57
    ALIGN = np.array([[match(a, b)[1].min() for b in range(NC)] for a in range(NC)])
    lab = -np.ones(NC, int)
    for c in range(NC):
        if lab[c] < 0:
            lab[c] = lab.max() + 1
            lab[(lab < 0) & (ALIGN[c] >= 0.9)] = lab[c]
    lp = stats["logp"].values.mean(axis=1)
    order = sorted(range(lab.max() + 1), key=lambda g: -lp[lab == g].mean())
    groups = [[c for c in range(NC) if lab[c] == g] for g in order]

    REF = groups[0][0]
    PERM = [match(REF, c)[0] for c in range(NC)]
    Ab = [np.mean([Amed[c][:, PERM[c]] for c in g], axis=0) for g in groups]
    Tb = [np.mean([THmed[c][:, PERM[c]] for c in g], axis=0) for g in groups]
    SDb = np.stack([post["theta"].values[g][..., PERM[g[0]]]
                    .reshape(-1, len(MOD), 3).std(axis=0) for g in groups]).max(axis=0)

    # axis identification: assign each probe bundle to the axis it loads on, in
    # the leading basin. Names, not top-loading benchmarks — GBAEval outranks
    # every classic item on the easy axis without being one.
    Pm = np.array([Ab[0][PIDX[k]].mean(axis=0) for k in PROBE])          # (3 probes, 3 axes)
    q = max(PERMS, key=lambda p: sum(Pm[i, p[i]] for i in range(3)))
    assert all(int(np.argmax(Pm[i])) == q[i] for i in range(3)), f"{tag}: probes not separable"
    AXIS = [None] * 3
    for i, k in enumerate(PROBE):
        AXIS[q[i]] = k

    fixed_d = np.array(json.loads(post.attrs["mirt_fixed_ceiling_d"])) \
        if "mirt_fixed_ceiling_d" in post.attrs else np.ones(len(BEN))

    def fit_mu(sel, cap=200):
        """Posterior-mean fitted values over the chains in `sel`, same link as the fit."""
        idx = np.linspace(0, post.sizes["draw"] - 1, min(cap, post.sizes["draw"])).astype(int)
        sub = post.isel(chain=sel, draw=idx)
        Av = sub["A"].values.reshape(-1, len(BEN), 3)
        Tv = sub["theta"].values.reshape(-1, len(MOD), 3)
        eta = -sub["D"].values.reshape(-1, len(BEN))[:, data.bench_idx]
        for k in range(3):
            eta += Av[:, data.bench_idx, k] * Tv[:, data.model_idx, k]
        c = FLOOR[data.bench_idx]
        d = (sub["ceiling_d"].values.reshape(-1, len(BEN))[:, data.bench_idx]
             if "ceiling_d" in sub else fixed_d[data.bench_idx])
        return (c + (d - c) * expit(eta)).mean(axis=0)

    def rm(mu):
        r = mu - YOBS
        return float(np.sqrt((r ** 2).mean())), float(np.abs(r).mean())

    GOF = {"all": rm(fit_mu(list(range(NC))))}
    for gi, g in enumerate(groups):
        GOF[f"basin {chr(65 + gi)}"] = rm(fit_mu(g))

    RH = {"pooled": mirt_identified_rhat(az.InferenceData(posterior=post), data)}
    for gi, g in enumerate(groups):
        RH[f"basin {chr(65 + gi)}"] = mirt_identified_rhat(
            az.InferenceData(posterior=post.isel(chain=g)), data)

    if tag == "A":
        # PROVE the 2 extra rows in A are the dropped GBAEval cells: rebuild their
        # Beta log-density from A's own posterior and match A's stored pointwise
        # log-likelihood. Order alone cannot do this (both rows sit in the
        # exact-zero tie block).
        ll2 = xr.open_dataset(path, group="log_likelihood")["obs"] \
                .isel(chain=0, draw=slice(None, None, THIN), obs_dim_0=DROP_A).load().values
        y2 = xr.open_dataset(path, group="observed_data")["obs"] \
               .isel(obs_dim_0=DROP_A).values
        gb = BEN.index("GBAEval")
        worst = 0.0
        for j, mn in enumerate(DROP_MODELS):
            mi = MOD.index(mn)
            eta = (post["A"].values[0, :, gb, :] * post["theta"].values[0, :, mi, :]).sum(1) \
                - post["D"].values[0, :, gb]
            mu = FLOOR[gb] + (1 - FLOOR[gb]) * expit(eta)
            phi = 1 / (4 * post["sigma_b"].values[0, :, gb] ** 2) - 1
            worst = max(worst, np.abs(beta_dist.logpdf(y2[j], mu * phi, (1 - mu) * phi)
                                      - ll2[:, j]).max())
        assert worst < 1e-8, worst
        print(f"A obs {DROP_A} identified as {DROP_MODELS} on GBAEval "
              f"(log-lik match to {worst:.1e}); subset for LOO")

    FITS[tag] = dict(NC=NC, groups=groups, ALIGN=ALIGN, lp=lp, Ab=Ab, Tb=Tb, SD=SDb,
                     AXIS=AXIS, RH=RH, GOF=GOF, fixed_d=fixed_d,
                     div=int(stats["diverging"].values.sum()),
                     ndraw=int(stats.sizes["chain"] * stats.sizes["draw"]),
                     n_obs=int(xr.open_dataset(path, group="observed_data").sizes["obs_dim_0"]),
                     gof_json=json.loads((path.parent / "mirt_gof_k3.json").read_text()))
    if tag == "B":
        FITS[tag]["lin_rate"] = post["lin_rate"].values.reshape(-1, 36, 3)
        FITS[tag]["vendors"] = json.loads(post.attrs["mirt_lineage_chains"])
    if tag == "C":
        FITS[tag]["ceiling_gap"] = post["ceiling_gap"].values.reshape(-1, len(BEN))
        FITS[tag]["ceiling_d"] = post["ceiling_d"].values.reshape(-1, len(BEN))
    del post, stats, Amed, THmed

DATES = pd.Series(MOD).map(_known_release_date_by_model())
XDATE = DATES.dt.strftime("%Y-%m-%d").values          # plotly-safe: no pd.Timestamp objects
HUMANS = set(json.loads(xr.open_dataset(PATHS["C"], group="posterior").attrs["mirt_human_order"]))
DATED = DATES.notna().values & ~np.isin(MOD, list(HUMANS))

print(f"{len(data.scores)} obs · {data.n_models} models · {data.n_benchmarks} benchmarks")
for t, f in FITS.items():
    print(f"{t}: {f['NC']} chains, sampled on {f['n_obs']} obs, basins "
          f"{[len(g) for g in f['groups']]}, logp "
          f"{[round(float(f['lp'][g].mean()), 1) for g in f['groups']]}, "
          f"{f['div']}/{f['ndraw']} div, axes {f['AXIS']}")

A obs [29, 30] identified as ['glm-5.1', 'MiniMax-M2.7'] on GBAEval (log-lik match to 1.3e-14); subset for LOO


4445 obs · 765 models · 98 benchmarks
A: 12 chains, sampled on 4447 obs, basins [6, 6], logp [3508.4, 3476.2], 15/36000 div, axes ['Easy knowledge', 'Fluid / abstract', 'Hard math + science']
B: 8 chains, sampled on 4445 obs, basins [4, 4], logp [3333.6, 3331.1], 5/16000 div, axes ['Fluid / abstract', 'Hard math + science', 'Easy knowledge']
C: 6 chains, sampled on 4445 obs, basins [4, 2], logp [3443.6, 3430.6], 18/12000 div, axes ['Hard math + science', 'Easy knowledge', 'Fluid / abstract']


### 1 · The 13 new benchmarks are thin: median **17** observations against **36** for the 85 they joined.
They are 8.4% of rows, and the 4 Epoch's own ECI uses are only **1.3%**. Seven of the 13 are majority post-2026 models, so they measure a narrow, recent slice.

In [2]:
grp = {b: (G_IN if b in NEW_IN else G_OUT) for b in NEW}
rows = []
for b in NEW:
    s = RAW[RAW["benchmark"] == b]
    dt = s["release_date"].dropna()
    rows.append(dict(bench=b, n=len(s), grp=grp[b], lo=dt.min(), hi=dt.max(),
                     lo_s=dt.min().strftime("%Y-%m-%d"), hi_s=dt.max().strftime("%Y-%m-%d"),
                     pct=100 * (dt >= pd.Timestamp("2026-01-01")).mean()))
T1 = pd.DataFrame(rows).sort_values("n")
MED_OLD = float(RAW[RAW["benchmark"].isin(OLD)].groupby("benchmark").size().median())
OLD_PCT = 100 * (RAW[RAW["benchmark"].isin(OLD)]["release_date"].dropna()
                 >= pd.Timestamp("2026-01-01")).mean()
names = T1["bench"].tolist()

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, column_widths=[0.30, 0.42, 0.28],
                    horizontal_spacing=0.035,
                    subplot_titles=["Observations", "Release-date span of the models tested",
                                    "% of rows after 2026-01-01"])
for g in (G_IN, G_OUT):
    s = T1[T1["grp"] == g]
    fig.add_trace(go.Bar(x=s["n"], y=s["bench"], orientation="h", name=g,
                         marker_color=GCOL[g], legendgroup=g,
                         text=s["n"], textposition="outside", textfont=dict(size=10),
                         hovertemplate="%{y}: %{x} obs<extra></extra>"), row=1, col=1)
    xs, ys = [], []
    for _, r in s.iterrows():
        xs += [r["lo_s"], r["hi_s"], None]; ys += [r["bench"], r["bench"], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", legendgroup=g, showlegend=False,
                             line=dict(color=GCOL[g], width=6), hoverinfo="skip"), row=1, col=2)
    fig.add_trace(go.Scatter(x=list(s["lo_s"]) + list(s["hi_s"]), y=list(s["bench"]) * 2,
                             mode="markers", legendgroup=g, showlegend=False,
                             marker=dict(color=GCOL[g], size=7, symbol="line-ns-open",
                                         line=dict(color=GCOL[g], width=2)),
                             hovertemplate="%{y}: %{x|%Y-%m-%d}<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Bar(x=s["pct"], y=s["bench"], orientation="h", legendgroup=g,
                         showlegend=False, marker_color=GCOL[g],
                         hovertemplate="%{y}: %{x:.0f}%<extra></extra>"), row=1, col=3)
fig.add_trace(go.Scatter(x=[MED_OLD] * 2, y=[names[0], names[-1]], mode="lines",
                         name=f"median of the 85 pre-existing ({MED_OLD:.0f} obs / {OLD_PCT:.0f}%)",
                         line=dict(color=C["dark"], width=1.6, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=[OLD_PCT] * 2, y=[names[0], names[-1]], mode="lines",
                         showlegend=False, line=dict(color=C["dark"], width=1.6, dash="dot")),
              row=1, col=3)
fig.update_xaxes(title_text="rows in the fit", range=[0, 118], row=1, col=1)
fig.update_xaxes(title_text="model release date", type="date", row=1, col=2)
fig.update_xaxes(title_text="% of rows", range=[0, 108], row=1, col=3)
fig.update_yaxes(categoryorder="array", categoryarray=names, tickfont=dict(size=10), row=1, col=1)
fig.update_layout(title="The 13 benchmarks added in the 2026-07-27/28 refresh",
                  height=470, width=1220, bargap=0.3,
                  legend=dict(orientation="h", y=-0.20), margin=dict(t=90, l=155))
show(fig, "01_new_benchmarks")

print(f"new: {T1['n'].sum()} rows = {100 * T1['n'].sum() / len(RAW):.1f}% · "
      f"in-ECI 4: {T1.loc[T1.grp == G_IN, 'n'].sum()} rows = "
      f"{100 * T1.loc[T1.grp == G_IN, 'n'].sum() / len(RAW):.1f}% · "
      f"median obs new {T1['n'].median():.0f} vs pre-existing {MED_OLD:.0f}")

new: 370 rows = 8.4% · in-ECI 4: 59 rows = 1.3% · median obs new 17 vs pre-existing 36


### 2 · GBAEval shares **zero** models with all 7 classic commonsense benchmarks, and with **46 of 85** pre-existing ones.
Seven of the 13 new benchmarks have no model overlap with more than half the old set (median 45 of 85). A zero-overlap pair has **no direct evidence** tying its loadings together — only a path through third benchmarks.

In [3]:
mods = {b: set(RAW.loc[RAW["benchmark"] == b, "model_version"]) for b in BEN}
OV = np.array([[len(mods[n] & mods[o]) for o in OLD] for n in NEW])
gb_classic = [int(len(mods["GBAEval"] & mods[c])) for c in CLASSIC]
assert sum(gb_classic) == 0, gb_classic
zeros = (OV == 0).sum(axis=1)
o_new = np.argsort(zeros)                                   # fewest zeros at the bottom
o_old = np.argsort([-OV[:, j].sum() for j in range(len(OLD))])
ylab = [f"<b>{NEW[i]}</b>" if NEW[i] in NEW_IN else NEW[i] for i in o_new]
# 4 legible bands instead of a continuous scale the eye cannot read
BINS, BCOL = [0, 1, 3, 10], [C["verm"], "#F6D6BE", C["sky"], C["blue"]]
BTXT = ["0 (none)", "1–2", "3–9", "10+"]
Z = np.digitize(OV[np.ix_(o_new, o_old)], BINS, right=False) - 1
scale = []
for i, col in enumerate(BCOL):
    scale += [[i / 4, col], [(i + 1) / 4, col]]

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.72, 0.28],
                    horizontal_spacing=0.03,
                    subplot_titles=["Shared models: 13 new benchmarks × 85 pre-existing",
                                    "Pre-existing benchmarks<br>with zero overlap"])
fig.add_trace(go.Heatmap(z=Z, x=[OLD[j] for j in o_old], y=ylab,
                         zmin=-0.5, zmax=3.5, colorscale=scale,
                         colorbar=dict(title=dict(text="shared models", side="right"),
                                       orientation="h", thickness=12, len=0.42, x=0.30,
                                       y=-0.42, tickvals=[0, 1, 2, 3], ticktext=BTXT),
                         customdata=OV[np.ix_(o_new, o_old)],
                         hovertemplate="%{y} × %{x}: %{customdata} shared<extra></extra>"),
              row=1, col=1)
for c in CLASSIC:
    fig.add_vline(x=list(o_old).index(OLD.index(c)), line=dict(color=C["dark"], width=1),
                  opacity=0.5, row=1, col=1)
fig.update_xaxes(tickmode="array",
                 tickvals=[list(o_old).index(OLD.index(c)) for c in CLASSIC],
                 ticktext=[f"<b>{c}</b>" for c in CLASSIC], tickangle=-90,
                 tickfont=dict(size=9.5), range=[-0.5, len(OLD) - 0.5],
                 title_text="85 pre-existing, sorted by total overlap; labelled ticks and "
                            "vertical rules = the 7 classic commonsense items", row=1, col=1)
fig.add_trace(go.Bar(x=zeros[o_new], y=ylab, orientation="h", marker_color=C["verm"],
                     name="count of zero-overlap partners", text=zeros[o_new],
                     textposition="outside", textfont=dict(size=10),
                     hovertemplate="%{y}: %{x} / 85 zero<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=[42.5] * 2, y=[ylab[0], ylab[-1]], mode="lines",
                         name="half of the 85", line=dict(color=C["dark"], width=1.6, dash="dot")),
              row=1, col=2)
fig.add_trace(go.Bar(x=[None], y=[None], marker_color="rgba(0,0,0,0)",
                     name="bold benchmark name = in Epoch's ECI"), row=1, col=2)
fig.update_xaxes(title_text="count / 85", range=[0, 96], row=1, col=2)
fig.update_yaxes(categoryorder="array", categoryarray=ylab, tickfont=dict(size=10),
                 row=1, col=1)
fig.update_layout(title="Zero model overlap: the new benchmarks are not co-measured with the old",
                  height=560, width=1250, bargap=0.3,
                  legend=dict(orientation="h", y=-0.56, x=0.62, xanchor="center"),
                  margin=dict(t=90, b=215, l=175))
show(fig, "02_overlap")

print(f"GBAEval: {len(mods['GBAEval'])} models · overlap with the 7 classic items "
      f"{gb_classic} · zero-overlap with {int(zeros[NEW.index('GBAEval')])} / 85")
print(f"{int((zeros > 42).sum())} of 13 new benchmarks miss more than half the old set; "
      f"median zero-overlap count {np.median(zeros):.0f}")

GBAEval: 14 models · overlap with the 7 classic items [0, 0, 0, 0, 0, 0, 0] · zero-overlap with 46 / 85
7 of 13 new benchmarks miss more than half the old set; median zero-overlap count 45


### 3 · GBAEval is near-separating: **7** models at or below 0.067, **7** at or above 0.316, nothing in between.
An item that splits the field cleanly is highly informative about whichever latent direction separates the two groups. Nothing in the data anchors **which** direction that is.

In [4]:
gs = RAW[RAW["benchmark"] == "GBAEval"].sort_values("score")
lo, hi = gs.loc[gs.score < 0.1, "score"].max(), gs.loc[gs.score > 0.1, "score"].min()
grpn = np.where(gs["score"].values <= lo, "low group (≤ 0.067)", "high group (≥ 0.316)")

fig = go.Figure()
fig.add_vrect(x0=lo, x1=hi, fillcolor=C["gray"], opacity=0.16, line_width=0,
              layer="below")
fig.add_trace(go.Scatter(x=[np.nan], y=[gs["model_version"].iloc[0]], mode="markers",
                         marker=dict(color=C["gray"], size=12, symbol="square", opacity=0.4),
                         name=f"empty interval {lo:.3f} → {hi:.3f}"))
for tag, col in [("low group (≤ 0.067)", C["verm"]), ("high group (≥ 0.316)", C["blue"])]:
    s = gs[grpn == tag]
    fig.add_trace(go.Scatter(x=s["score"], y=s["model_version"], mode="markers",
                             name=f"{tag}, n={len(s)}",
                             marker=dict(color=col, size=11, line=dict(color="white", width=1)),
                             hovertemplate="%{y}: %{x:.4f}<extra></extra>"))
fig.add_trace(go.Scatter(x=[np.median(gs["score"])] * 2,
                         y=[gs["model_version"].iloc[0], gs["model_version"].iloc[-1]],
                         mode="lines", name=f"median {np.median(gs['score']):.3f}",
                         line=dict(color=C["dark"], width=1.6, dash="dot")))
fig.update_xaxes(title_text="GBAEval overall score", range=[-0.03, 0.80])
fig.update_yaxes(categoryorder="array", categoryarray=gs["model_version"].tolist(),
                 tickfont=dict(size=10))
fig.update_layout(title="GBAEval score distribution: a near-binary split, no middle",
                  height=480, width=880, legend=dict(orientation="h", y=-0.16),
                  margin=dict(l=210, t=80))
show(fig, "03_gbaeval_scores")

print(f"n={len(gs)} · ≤{lo:.4f}: {int((gs.score <= lo).sum())} · "
      f"≥{hi:.4f}: {int((gs.score >= hi).sum())} · median {np.median(gs.score):.4f} "
      f"(sits inside the empty interval)")

n=14 · ≤0.0670: 7 · ≥0.3160: 7 · median 0.1915 (sits inside the empty interval)


### 4 · The 2 dropped rows are harness failures, not incapability: **143 of ~145** tasks failed grading.
They are the only 2 of 16 GBAEval rows with any recorded failure, and all three sub-scores are exactly 0. glm-5.2 scores **0.000172** with no failures — that is what real incapability looks like here.

In [5]:
SNAP = ROOT / "data/pipeline/snapshots/2026-07-28/epoch/gbaeval_external.csv"
raw = pd.read_csv(SNAP).sort_values("Overall score")
drops = pd.read_csv(ROOT / "data/curated/row_drops_2026-07-28.csv")
assert set(drops["model_version"]) == set(DROP_MODELS) and len(raw) == 16
raw["fail"] = raw["Failure total"].fillna(0)
ylab = [f"<b>{m}</b>" if m in DROP_MODELS else m for m in raw["Model version"]]

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.55, 0.45],
                    horizontal_spacing=0.05,
                    subplot_titles=["Reported scores", "Recorded harness failures (tasks)"])
for tag, colname, col, sym in [("overall", "Overall score", C["dark"], "diamond"),
                               ("Replay", "Replay score", C["blue"], "circle"),
                               ("Procedural", "Procedural score", C["sky"], "square"),
                               ("Audio", "Audio score", C["green"], "triangle-up")]:
    fig.add_trace(go.Scatter(x=raw[colname], y=ylab, mode="markers", name=tag,
                             marker=dict(color=col, size=10 if tag == "overall" else 8,
                                         symbol=sym, line=dict(color="white", width=1)),
                             hovertemplate="%{y}: %{x:.4f}<extra></extra>"), row=1, col=1)
for tag, colname, col in [("Build failed", "Build failed", C["orange"]),
                          ("Grading failed", "Grading failed", C["verm"])]:
    fig.add_trace(go.Bar(x=raw[colname].fillna(0), y=ylab, orientation="h", name=tag,
                         marker_color=col, hovertemplate="%{y}: %{x:.0f}<extra></extra>"),
                  row=1, col=2)
fig.add_trace(go.Scatter(x=[145] * 2, y=[ylab[0], ylab[-1]], mode="lines",
                         name="~145 tasks attempted",
                         line=dict(color=C["dark"], width=1.6, dash="dot")), row=1, col=2)
fig.update_xaxes(title_text="score", range=[-0.03, 1.0], row=1, col=1)
fig.update_xaxes(title_text="tasks", range=[0, 168], row=1, col=2)
fig.update_yaxes(categoryorder="array", categoryarray=ylab, tickfont=dict(size=10), row=1, col=1)
fig.add_trace(go.Bar(x=[None], y=[None], marker_color="rgba(0,0,0,0)",
                     name="bold = dropped 2026-07-28"), row=1, col=2)
fig.update_layout(title="GBAEval raw rows: the two zeros are the only rows that failed",
                  height=520, width=1120, barmode="stack", bargap=0.3,
                  legend=dict(orientation="h", y=-0.16), margin=dict(l=200, t=90))
show(fig, "04_dropped_rows")

cols = ["Model version", "Overall score", "Replay score", "Procedural score", "Audio score",
        "Build failed", "Grading failed", "Failure total", "Checkpoints"]
print(raw.loc[raw["Model version"].isin(DROP_MODELS + ["glm-5.2_unknown"]), cols].to_string(index=False))
print(f"rows with any recorded failure: {int((raw['fail'] > 0).sum())} / {len(raw)}")

  Model version  Overall score  Replay score  Procedural score  Audio score  Build failed  Grading failed  Failure total  Checkpoints
        glm-5.1       0.000000      0.000000      0.000000e+00          0.0           2.0           143.0          145.0            0
   MiniMax-M2.7       0.000000      0.000000      0.000000e+00          0.0           1.0           143.0          144.0            0
glm-5.2_unknown       0.000172      0.000287      9.999999e-17          0.0           NaN             NaN            NaN           96
rows with any recorded failure: 2 / 16


### 5 · B and C converge equally badly and C fits better: identified eta r̂ **1.60** vs **1.55**, RMSE **0.0471** vs **0.0458**.
Divergences are low in both (**5 / 16,000** and **18 / 12,000**). Neither BM variant is a convergence fix.

In [6]:
KEYS = ["eta_max_rhat", "D_max_rhat", "sigma_b_max_rhat"]
KLAB = ["eta (max)", "D (max)", "sigma_b (max)"]
fig = make_subplots(rows=1, cols=3, column_widths=[0.40, 0.22, 0.38], horizontal_spacing=0.10,
                    subplot_titles=["Identified r̂ (pooled over all chains)",
                                    "Divergences per 10,000 draws",
                                    "Goodness of fit (mirt_gof_k3.json)"])
for t in ("B", "C"):
    f = FITS[t]
    v = [float(f["RH"]["pooled"][k]) for k in KEYS]
    fig.add_trace(go.Bar(x=KLAB, y=v, name=SHORT[t], legendgroup=t, marker_color=FCOL[t],
                         text=[f"{x:.3f}" for x in v], textposition="outside",
                         textfont=dict(size=10)), row=1, col=1)
    dv = 1e4 * f["div"] / f["ndraw"]
    fig.add_trace(go.Bar(x=["divergent"], y=[dv], legendgroup=t, showlegend=False,
                         marker_color=FCOL[t], text=[f"{f['div']}/{f['ndraw']}"],
                         textposition="outside", textfont=dict(size=10)), row=1, col=2)
    g = f["gof_json"]
    fig.add_trace(go.Bar(x=["RMSE", "MAE", "1 − R²"],
                         y=[g["rmse"], g["mae"], 1 - g["bayesian_r2"]], legendgroup=t,
                         showlegend=False, marker_color=FCOL[t],
                         text=[f"{g['rmse']:.4f}", f"{g['mae']:.4f}", f"{g['bayesian_r2']:.4f}"],
                         textposition="outside", textfont=dict(size=10)), row=1, col=3)
fig.add_trace(go.Scatter(x=KLAB, y=[1.01] * 3, mode="lines",
                         name="r̂ = 1.01 (convergence bar)",
                         line=dict(color=C["dark"], width=1.5, dash="dot")), row=1, col=1)
fig.update_yaxes(title_text="r̂", range=[1.0, 1.82], row=1, col=1)
fig.update_yaxes(title_text="per 10,000", range=[0, 22], row=1, col=2)
fig.update_yaxes(title_text="error (bar text: R² for the third bar)", range=[0, 0.062], row=1, col=3)
fig.update_layout(title="The two Brownian fits, side by side", height=470, width=1220,
                  bargap=0.3, legend=dict(orientation="h", y=-0.16), margin=dict(t=100))
show(fig, "05_B_vs_C_convergence")

for t in ("B", "C"):
    f, g = FITS[t], FITS[t]["gof_json"]
    print(f"{t}: eta r̂ {float(f['RH']['pooled']['eta_max_rhat']):.3f} "
          f"(mean {float(f['RH']['pooled']['eta_mean_rhat']):.3f}, "
          f"frac>1.01 {float(f['RH']['pooled']['eta_frac_gt_1.01']):.2f}) · "
          f"D {float(f['RH']['pooled']['D_max_rhat']):.3f} · "
          f"sigma_b {float(f['RH']['pooled']['sigma_b_max_rhat']):.3f} · "
          f"{f['div']}/{f['ndraw']} div · RMSE {g['rmse']:.6f} MAE {g['mae']:.6f} "
          f"R² {g['bayesian_r2']:.6f} on {g['n_obs']} obs")

B: eta r̂ 1.601 (mean 1.032, frac>1.01 0.50) · D 1.690 · sigma_b 1.511 · 5/16000 div · RMSE 0.047111 MAE 0.031660 R² 0.955174 on 4445 obs
C: eta r̂ 1.551 (mean 1.031, frac>1.01 0.52) · D 1.624 · sigma_b 1.500 · 18/12000 div · RMSE 0.045823 MAE 0.030620 R² 0.957421 on 4445 obs


### 6 · The rate hierarchy is empty and the noise gap fires on saturated old benchmarks, not new ones.
All 36 vendor rates in B sit inside **9.2 / 20.1 / 8.7 %** of one vendor's own 90% interval. In C only **3 of 98** benchmarks pull a gap above 0.10: SWE-Bench Verified, WMDP Biology, MMLU.

In [7]:
f = FITS["B"]
lr, VEND = f["lin_rate"], f["vendors"]
med = np.median(lr, axis=0)                                   # (36, 3)
ciw = np.diff(np.quantile(lr, [0.05, 0.95], axis=0), axis=0)[0].mean(axis=0)
gap = FITS["C"]["ceiling_gap"]
gm, gq = np.median(gap, axis=0), np.quantile(gap, [0.05, 0.95], axis=0)
top = np.argsort(-gm)[:10][::-1]
AXB = FITS["B"]["AXIS"]

fig = make_subplots(rows=1, cols=2, column_widths=[0.50, 0.50], horizontal_spacing=0.13,
                    subplot_titles=["B · 36 vendor climb rates vs one vendor's own 90% interval",
                                    "C · estimated ceiling gap, 10 largest of 98"])
for k in range(3):
    lo, hi = med[:, k].mean() - ciw[k] / 2, med[:, k].mean() + ciw[k] / 2
    fig.add_trace(go.Scatter(x=[lo, hi, hi, lo, lo], y=[k - .3, k - .3, k + .3, k + .3, k - .3],
                             mode="lines", fill="toself", fillcolor="rgba(153,153,153,0.22)",
                             line=dict(color=C["gray"], width=1),
                             name="mean 90% interval of a single vendor" if k == 0 else None,
                             showlegend=k == 0, hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=med[:, k], y=[k] * 36, mode="markers",
                             name="vendor posterior median (36 chains)" if k == 0 else None,
                             showlegend=k == 0,
                             marker=dict(color=C["orange"], size=8, symbol="line-ns-open",
                                         line=dict(color=C["orange"], width=2)),
                             text=VEND, hovertemplate="%{text}: %{x:.3f}<extra></extra>"),
                  row=1, col=1)
    sp = med[:, k].max() - med[:, k].min()
    fig.add_annotation(x=med[:, k].mean(), y=k + 0.42, showarrow=False, row=1, col=1,
                       font=dict(size=10, color=C["dark"]),
                       text=f"spread {sp:.3f} = {100 * sp / ciw[k]:.0f}% of the interval")
fig.update_yaxes(tickmode="array", tickvals=list(range(3)), ticktext=AXB,
                 range=[-0.6, 2.75], row=1, col=1)
fig.update_xaxes(title_text="climb rate (logits / year)", row=1, col=1)

fig.add_trace(go.Scatter(x=gm[top], y=[BEN[i] for i in top], mode="markers",
                         name="posterior median (5–95%)",
                         marker=dict(color=C["blue"], size=10, line=dict(color="white", width=1)),
                         error_x=dict(type="data", symmetric=False, array=gq[1][top] - gm[top],
                                      arrayminus=gm[top] - gq[0][top], color=C["blue"],
                                      thickness=1.4, width=5),
                         hovertemplate="%{y}: %{x:.3f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=[0.02] * 2, y=[BEN[top[0]], BEN[top[-1]]], mode="lines",
                         name="prior mean, Beta(1, 49) = 0.02",
                         line=dict(color=C["dark"], width=1.6, dash="dot")), row=1, col=2)
fig.update_xaxes(title_text="ceiling gap  (d = fixed_d × (1 − gap))", range=[0, 0.24], row=1, col=2)
fig.update_yaxes(tickfont=dict(size=10), row=1, col=2)
fig.update_layout(title="Both extra structures are near-empty", height=470, width=1250,
                  legend=dict(orientation="h", y=-0.20), margin=dict(t=100, l=110))
show(fig, "06_hierarchy_and_gap")

print("B lin_rate: " + " · ".join(
    f"{AXB[k]} mean {med[:, k].mean():.3f}, vendor sd {med[:, k].std():.4f}, "
    f"spread/CI {100 * (med[:, k].max() - med[:, k].min()) / ciw[k]:.1f}%" for k in range(3)))
print(f"C ceiling_gap: median over benchmarks {np.median(gm):.4f} · "
      f"{int((gm > 0.05).sum())} above 0.05 · {int((gm > 0.10).sum())} above 0.10 → "
      + ", ".join(f"{BEN[i]} {gm[i]:.3f}" for i in np.argsort(-gm)[:3]))
print("  fixed 4PL ceilings: " + ", ".join(
    f"{BEN[i]} d={FITS['C']['fixed_d'][i]:.2f}" for i in np.where(FITS["C"]["fixed_d"] < 1)[0]))

B lin_rate: Fluid / abstract mean 0.667, vendor sd 0.0126, spread/CI 9.2% · Hard math + science mean 0.493, vendor sd 0.0398, spread/CI 20.1% · Easy knowledge mean 0.691, vendor sd 0.0201, spread/CI 8.7%
C ceiling_gap: median over benchmarks 0.0153 · 8 above 0.05 · 3 above 0.10 → SWE-Bench Verified 0.175, WMDP Biology 0.159, MMLU 0.140
  fixed 4PL ceilings: FrontierMath Tier 4 v1 d=0.60, FrontierMath v1 d=0.57


### 7 · B and C agree: loadings correlate **0.98 / 0.98 / 0.97** per axis in their leading basins.
The rate hierarchy and the noise gap change the parameter count, not the measurement frame. The rest of the notebook therefore uses **C** — simpler and better-fitting — as the Brownian representative against **A**.

In [8]:
AB, AC = FITS["B"]["Ab"][0], FITS["C"]["Ab"][0]
M = np.corrcoef(AB.T, AC.T)[:3, 3:]
q = max(PERMS, key=lambda p: sum(M[i, p[i]] for i in range(3)))
r = [M[i, q[i]] for i in range(3)]
AXB, AXC = FITS["B"]["AXIS"], FITS["C"]["AXIS"]
assert [AXC[q[i]] for i in range(3)] == AXB, (AXB, [AXC[j] for j in q])

fig = go.Figure()
lim = 1.05 * max(AB.max(), AC.max())
fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode="lines", name="identity",
                         line=dict(color=C["dark"], width=1.2, dash="dot"), hoverinfo="skip"))
for i, col in enumerate([C["blue"], C["orange"], C["green"]]):
    lab = [f"<b>{b}</b>" if b in NEW else b for b in BEN]
    fig.add_trace(go.Scatter(x=AB[:, i], y=AC[:, q[i]], mode="markers",
                             name=f"{AXB[i]} · r = {r[i]:.3f}",
                             marker=dict(color=col, size=8, opacity=0.75,
                                         line=dict(color="white", width=0.7)),
                             text=lab, hovertemplate="%{text}<br>B %{x:.2f} → C %{y:.2f}<extra></extra>"))
gi = BEN.index("GBAEval")
fig.add_annotation(x=AB[gi, AXB.index("Easy knowledge")],
                   y=AC[gi, AXC.index("Easy knowledge")], text="GBAEval", showarrow=True,
                   arrowhead=2, ax=-46, ay=-24, font=dict(size=11, color=C["dark"]),
                   arrowcolor=C["dark"])
fig.update_layout(title=f"Per-benchmark loadings, fit B versus fit C (leading basin of each) · "
                        f"flat r = {np.corrcoef(AB.ravel(), AC[:, list(q)].ravel())[0, 1]:.3f}",
                  xaxis=dict(title="loading in B (BM + rate hierarchy)", range=[0, lim]),
                  yaxis=dict(title="loading in C (BM + noise gap)", range=[0, lim],
                             scaleanchor="x", scaleratio=1),
                  height=560, width=760, legend=dict(orientation="h", y=-0.15))
show(fig, "07_B_vs_C_loadings")

print("axis order B→C:", [(AXB[i], round(r[i], 3)) for i in range(3)])
print(f"RMSE B {FITS['B']['gof_json']['rmse']:.5f} vs C {FITS['C']['gof_json']['rmse']:.5f}; "
      f"C has {len(FITS['B']['lin_rate'][0].ravel())} fewer vendor-rate parameters")

axis order B→C: [('Fluid / abstract', np.float64(0.979)), ('Hard math + science', np.float64(0.981)), ('Easy knowledge', np.float64(0.971))]
RMSE B 0.04711 vs C 0.04582; C has 108 fewer vendor-rate parameters


### 8 · Every fit splits into two basins. Brownian steps do not merge them.
A splits 6/6 at **32.2** nats, B 4/4 at **2.5**, C 4/2 at **13.0**; worst cross-basin alignment **0.48 / 0.37 / 0.56**.
A's two basins are each internally converged (identified eta r̂ **1.004** apiece). B's and C's leading basins are not (**1.46** and **1.48**), so their mode structure is finer than two.

In [9]:
fig = make_subplots(rows=2, cols=3, row_heights=[0.42, 0.58], vertical_spacing=0.16,
                    horizontal_spacing=0.11,
                    subplot_titles=[
                        f"{SHORT[t]} · gap "
                        f"{FITS[t]['lp'][FITS[t]['groups'][0]].mean() - FITS[t]['lp'][FITS[t]['groups'][1]].mean():.1f}"
                        " nats" for t in ("A", "B", "C")] +
                        ["Chain-pair alignment (worst matched axis)", "", ""])
for ci_, t in enumerate(("A", "B", "C"), 1):
    f = FITS[t]
    ORD = f["groups"][0] + f["groups"][1]
    tick = [f"c{c}" for c in ORD]
    for gi, (g, col) in enumerate(zip(f["groups"], [C["blue"], C["verm"]])):
        fig.add_trace(go.Scatter(x=[f"c{c}" for c in g], y=f["lp"][g], mode="markers",
                                 name=f"basin {chr(65 + gi)} (dotted line = its mean logp)",
                                 legendgroup=f"b{gi}",
                                 showlegend=ci_ == 1,
                                 marker=dict(color=col, size=10, symbol="diamond",
                                             line=dict(color="white", width=1)),
                                 hovertemplate="%{x}: %{y:.1f}<extra></extra>"), row=1, col=ci_)
        fig.add_hline(y=f["lp"][g].mean(), line=dict(color=col, width=1, dash="dot"),
                      row=1, col=ci_)
    fig.add_trace(go.Heatmap(z=f["ALIGN"][np.ix_(ORD, ORD)], x=tick, y=tick,
                             zmin=0.3, zmax=1.0, showscale=ci_ == 3,
                             colorscale=[[0, "#FFF7EC"], [0.5, C["orange"]], [1.0, C["blue"]]],
                             colorbar=dict(title="worst matched<br>axis corr", thickness=12,
                                           len=0.44, x=1.005, y=0.22),
                             hovertemplate="%{y} vs %{x}: %{z:.3f}<extra></extra>"),
                  row=2, col=ci_)
    n0 = len(f["groups"][0]) - 0.5
    fig.add_vline(x=n0, line=dict(color=C["dark"], width=2), row=2, col=ci_)
    fig.add_hline(y=n0, line=dict(color=C["dark"], width=2), row=2, col=ci_)
    fig.update_xaxes(categoryorder="array", categoryarray=tick, title_text="chain",
                     tickfont=dict(size=9), row=1, col=ci_)
    fig.update_xaxes(title_text="chain (basin A left)", tickfont=dict(size=9), row=2, col=ci_)
    fig.update_yaxes(autorange="reversed", tickfont=dict(size=9), row=2, col=ci_)
fig.update_yaxes(title_text="mean logp", row=1, col=1)
fig.update_layout(title="Two basins in all three fits", height=780, width=1250,
                  legend=dict(orientation="h", y=-0.10), margin=dict(t=90, b=110, r=110))
show(fig, "08_basin_census")

for t in ("A", "B", "C"):
    f = FITS[t]
    ga, gb_ = f["groups"]
    print(f"{t}: basins {ga} / {gb_} · logp {f['lp'][ga].mean():.1f} vs {f['lp'][gb_].mean():.1f} "
          f"(gap {f['lp'][ga].mean() - f['lp'][gb_].mean():.1f}) · worst cross "
          f"{f['ALIGN'][np.ix_(ga, gb_)].min():.3f} · worst within "
          f"{min(f['ALIGN'][np.ix_(g, g)].min() for g in f['groups']):.3f} · eta r̂ pooled "
          f"{float(f['RH']['pooled']['eta_max_rhat']):.3f} → per basin "
          f"{[round(float(f['RH'][f'basin {chr(65 + i)}']['eta_max_rhat']), 3) for i in range(2)]}")

A: basins [1, 3, 5, 6, 8, 10] / [0, 2, 4, 7, 9, 11] · logp 3508.4 vs 3476.2 (gap 32.2) · worst cross 0.482 · worst within 0.999 · eta r̂ pooled 1.616 → per basin [1.004, 1.004]
B: basins [2, 3, 4, 5] / [0, 1, 6, 7] · logp 3333.6 vs 3331.1 (gap 2.5) · worst cross 0.368 · worst within 0.928 · eta r̂ pooled 1.601 → per basin [1.461, 1.009]
C: basins [0, 2, 3, 5] / [1, 4] · logp 3443.6 vs 3430.6 (gap 13.0) · worst cross 0.564 · worst within 0.912 · eta r̂ pooled 1.551 → per basin [1.482, 1.048]


### 9 · The split is the same question in all three fits: does GBAEval load on easy knowledge or on the fluid axis?
In the leading basin it tops the easy axis at **2.4** in every fit; in the other basin it drops to **0.2** and reappears on the fluid axis at ~2.1. Axes are named by the mean loading of the 7 classic items, never by the top-loading benchmark.

In [10]:
ORDER = ["Easy knowledge", "Hard math + science", "Fluid / abstract"]
fig = make_subplots(rows=3, cols=3, horizontal_spacing=0.10, vertical_spacing=0.075,
                    subplot_titles=[f"{SHORT[t]} · {a}" for t in ("A", "B", "C") for a in ORDER])
for ri, t in enumerate(("A", "B", "C"), 1):
    f = FITS[t]
    for ci_, aname in enumerate(ORDER, 1):
        k = f["AXIS"].index(aname)
        A0, A1 = f["Ab"][0][:, k], f["Ab"][1][:, k]
        sel = np.argsort(-np.maximum(A0, A1))[:8][::-1]
        lab = [f"<b>{BEN[i]}</b>" if BEN[i] in NEW else BEN[i] for i in sel]
        for gi, (v, col) in enumerate([(A0[sel], C["blue"]), (A1[sel], C["verm"])]):
            fig.add_trace(go.Bar(x=v, y=lab, orientation="h", marker_color=col,
                                 name=f"basin {chr(65 + gi)}", legendgroup=f"b{gi}",
                                 showlegend=(ri == 1 and ci_ == 1),
                                 hovertemplate="%{y}: %{x:.2f}<extra></extra>"), row=ri, col=ci_)
        fig.update_xaxes(range=[0, 1.06 * max(A0.max(), A1.max())], tickfont=dict(size=9),
                         title_text="loading" if ri == 3 else None, row=ri, col=ci_)
        fig.update_yaxes(tickfont=dict(size=8.5), row=ri, col=ci_)
fig.add_trace(go.Bar(x=[None], y=[None], marker_color="rgba(0,0,0,0)",
                     name="bold = added in the 2026-07-27/28 refresh"), row=1, col=1)
fig.update_layout(title="Top 8 loadings per axis, per basin, per fit", height=1000, width=1250,
                  barmode="group", bargap=0.24, legend=dict(orientation="h", y=1.06),
                  margin=dict(t=130, l=150))
show(fig, "09_axis_loadings")

gi_ = BEN.index("GBAEval")
for t in ("A", "B", "C"):
    f = FITS[t]
    ke, kf = f["AXIS"].index("Easy knowledge"), f["AXIS"].index("Fluid / abstract")
    cm = [f["Ab"][b][[BEN.index(c) for c in CLASSIC]].mean(axis=0) for b in (0, 1)]
    best = max([BEN.index(c) for c in CLASSIC], key=lambda i: f["Ab"][0][i, ke])
    print(f"{t}: classic-item mean per axis, basin A {np.round(cm[0], 2)} / basin B "
          f"{np.round(cm[1], 2)} · GBAEval easy-axis {f['Ab'][0][gi_, ke]:.2f} → "
          f"{f['Ab'][1][gi_, ke]:.2f}, fluid-axis {f['Ab'][0][gi_, kf]:.2f} → "
          f"{f['Ab'][1][gi_, kf]:.2f}")
    print(f"    strongest classic item on the easy axis: {BEN[best]} "
          f"{f['Ab'][0][best, ke]:.2f} → GBAEval is "
          f"{100 * (f['Ab'][0][gi_, ke] / f['Ab'][0][best, ke] - 1):+.0f}% above it, on a "
          f"benchmark categorised {RAW.loc[RAW.benchmark == 'GBAEval', 'category'].iloc[0]!r} "
          f"with zero model overlap with any classic item")

A: classic-item mean per axis, basin A [1.32 0.22 0.26] / basin B [1.27 0.23 0.22] · GBAEval easy-axis 2.47 → 0.15, fluid-axis 0.22 → 2.16
    strongest classic item on the easy axis: ARC (AI2) 1.74 → GBAEval is +42% above it, on a benchmark categorised 'Agentic Computer Use' with zero model overlap with any classic item
B: classic-item mean per axis, basin A [0.29 0.22 1.28] / basin B [0.22 0.22 1.28] · GBAEval easy-axis 2.42 → 0.16, fluid-axis 0.28 → 2.11
    strongest classic item on the easy axis: GSM8K 1.84 → GBAEval is +31% above it, on a benchmark categorised 'Agentic Computer Use' with zero model overlap with any classic item
C: classic-item mean per axis, basin A [0.26 1.39 0.33] / basin B [0.28 1.36 0.26] · GBAEval easy-axis 2.37 → 0.19, fluid-axis 0.27 → 2.20
    strongest classic item on the easy axis: GSM8K 2.15 → GBAEval is +10% above it, on a benchmark categorised 'Agentic Computer Use' with zero model overlap with any classic item


### 10 · The basins disagree exactly where few models are measured, in all three fits alike.
Hard math + science has **9 / 42 / 69** measured models and cross-basin ability correlation **0.97 / 0.98 / 0.80**. Easy knowledge has only **13 / 10 / 10**, and there the two basins *anti*-correlate (**−0.47 / −0.77 / −0.07**).
A model counts as measured only if θ SD < 0.3 in **both** basins, which is why the two thin axes keep so few.

In [11]:
fig = make_subplots(rows=3, cols=3, shared_xaxes=False, horizontal_spacing=0.055,
                    vertical_spacing=0.085,
                    subplot_titles=[f"{SHORT[t]} · {a}" for t in ("A", "B", "C") for a in ORDER])
for ri, t in enumerate(("A", "B", "C"), 1):
    f = FITS[t]
    for ci_, aname in enumerate(ORDER, 1):
        k = f["AXIS"].index(aname)
        m = DATED & (f["SD"][:, k] < 0.3)          # measured, not extrapolated; humans out
        x = XDATE[m]
        o = np.argsort(DATES[m].values)
        for gi, col in enumerate([C["blue"], C["verm"]]):
            y = f["Tb"][gi][m, k]
            fig.add_trace(go.Scatter(x=x[o], y=y[o], mode="markers",
                                     name=f"basin {chr(65 + gi)} models",
                                     legendgroup=f"b{gi}", showlegend=(ri == 1 and ci_ == 1),
                                     marker=dict(color=col, size=4, opacity=0.5),
                                     text=np.array(MOD)[m][o],
                                     hovertemplate="%{text}<br>%{y:.2f}<extra></extra>"),
                          row=ri, col=ci_)
            fig.add_trace(go.Scatter(x=x[o], y=np.maximum.accumulate(y[o]), mode="lines",
                                     name=f"basin {chr(65 + gi)} running frontier",
                                     legendgroup=f"f{gi}", showlegend=(ri == 1 and ci_ == 1),
                                     line=dict(color=col, width=2.4)), row=ri, col=ci_)
        r_ = np.corrcoef(f["Tb"][0][m, k], f["Tb"][1][m, k])[0, 1]
        fig.add_annotation(x=0.5, y=1.03, xref="x domain", yref="y domain", showarrow=False,
                           text=f"n={int(m.sum())} measured · corr(A,B)={r_:.2f}",
                           font=dict(size=9.5, color=C["dark"]), row=ri, col=ci_)
        fig.update_xaxes(tickfont=dict(size=9), row=ri, col=ci_)
        fig.update_yaxes(tickfont=dict(size=9), row=ri, col=ci_)
    fig.update_yaxes(title_text="ability (logits)", row=ri, col=1)
fig.update_layout(title="Ability timeline per axis and basin (measured models only, θ SD < 0.3)",
                  height=1000, width=1250, legend=dict(orientation="h", y=-0.055),
                  margin=dict(t=90, b=110))
show(fig, "10_timelines")

for t in ("A", "B", "C"):
    f = FITS[t]
    out = []
    for aname in ORDER:
        k = f["AXIS"].index(aname)
        m = DATED & (f["SD"][:, k] < 0.3)
        out.append(f"{aname} n={int(m.sum())} corr={np.corrcoef(f['Tb'][0][m, k], f['Tb'][1][m, k])[0, 1]:.2f}")
    print(f"{t}: " + " · ".join(out))

A: Easy knowledge n=13 corr=-0.47 · Hard math + science n=9 corr=0.97 · Fluid / abstract n=13 corr=0.80
B: Easy knowledge n=10 corr=-0.77 · Hard math + science n=42 corr=0.98 · Fluid / abstract n=10 corr=0.89
C: Easy knowledge n=10 corr=-0.07 · Hard math + science n=69 corr=0.80 · Fluid / abstract n=8 corr=0.95


### 11 · C fits best on every split: RMSE **0.0465** against A's **0.0473** and B's **0.0477**; the basins differ by under 0.0013.
R² is only comparable across fits whose observation sets match. A was sampled on 4,447 rows, so here every residual is recomputed on the **common 4,445** — which makes the comparison safe rather than assuming it.

In [12]:
SPL = ["all", "basin A", "basin B"]
SCOL = {"all": C["dark"], "basin A": C["blue"], "basin B": C["verm"]}
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=["RMSE", "MAE"])
for j, mi in enumerate([0, 1]):
    for s in SPL:
        v = [FITS[t]["GOF"][s][mi] for t in ("A", "B", "C")]
        fig.add_trace(go.Bar(x=[SHORT[t] for t in ("A", "B", "C")], y=v, name=s,
                             legendgroup=s, showlegend=j == 0, marker_color=SCOL[s],
                             text=[f"{x:.4f}" for x in v], textposition="outside",
                             textfont=dict(size=9.5),
                             hovertemplate="%{x}: %{y:.5f}<extra></extra>"), row=1, col=j + 1)
    fig.update_yaxes(range=[0, 1.28 * max(FITS[t]["GOF"][s][mi]
                                          for t in FITS for s in SPL)], row=1, col=j + 1)
fig.update_yaxes(title_text="RMSE", row=1, col=1)
fig.update_yaxes(title_text="MAE", row=1, col=2)
fig.update_layout(title="Fit error on the common 4,445 observations, pooled and per basin",
                  height=470, width=1050, bargap=0.28,
                  legend=dict(orientation="h", y=-0.16), margin=dict(t=100))
show(fig, "11_rmse_mae")

for t in ("A", "B", "C"):
    g = FITS[t]["gof_json"]
    print(f"{t}: recomputed " + " · ".join(
        f"{s} RMSE {FITS[t]['GOF'][s][0]:.5f} MAE {FITS[t]['GOF'][s][1]:.5f}" for s in SPL))
    print(f"    stored mirt_gof_k3.json (posterior-predictive draws, "
          f"{g['n_obs']} obs): RMSE {g['rmse']:.5f} MAE {g['mae']:.5f} R² {g['bayesian_r2']:.5f}")
print("recomputed values use posterior-mean fitted values, so they sit ~0.0006 above the "
      "stored draw-based RMSE; the ordering across fits is identical.")

A: recomputed all RMSE 0.04737 MAE 0.03207 · basin A RMSE 0.04796 MAE 0.03236 · basin B RMSE 0.04830 MAE 0.03248
    stored mirt_gof_k3.json (posterior-predictive draws, 4447 obs): RMSE 0.04671 MAE 0.03172 R² 0.95561
B: recomputed all RMSE 0.04768 MAE 0.03196 · basin A RMSE 0.04777 MAE 0.03203 · basin B RMSE 0.04881 MAE 0.03247
    stored mirt_gof_k3.json (posterior-predictive draws, 4445 obs): RMSE 0.04711 MAE 0.03166 R² 0.95517
C: recomputed all RMSE 0.04644 MAE 0.03091 · basin A RMSE 0.04676 MAE 0.03105 · basin B RMSE 0.04734 MAE 0.03146
    stored mirt_gof_k3.json (posterior-predictive draws, 4445 obs): RMSE 0.04582 MAE 0.03062 R² 0.95742
recomputed values use posterior-mean fitted values, so they sit ~0.0006 above the stored draw-based RMSE; the ordering across fits is identical.


### 12 · LOO ranks C > B > A by **+99.7 ± 29.2** and **+26.1 ± 13.8** nats — but PSIS is unreliable here.
Fit A's log-likelihood is subset to the common 4,445 columns (the 2 dropped GBAEval rows, identified from A's own log-density to 1e-14), so the elpd difference is over one shared observation set. **p_loo ≈ 1,600** on 4,445 points and **704–728** observations at k > 0.7: read the ranking, not the magnitude.

In [13]:
loos, PW = {}, {}
for t, path in PATHS.items():
    ll = xr.open_dataset(path, group="log_likelihood").isel(draw=slice(None, None, THIN)).load()
    if t == "A":
        keep = np.setdiff1d(np.arange(ll.sizes["obs_dim_0"]), DROP_A)
        ll = ll.isel(obs_dim_0=keep).assign_coords(obs_dim_0=np.arange(len(keep)))
    assert ll.sizes["obs_dim_0"] == len(data.scores)
    tiny = xr.open_dataset(path, group="posterior")[["tau_A"]] \
             .isel(draw=slice(None, None, THIN)).load()
    loos[SHORT[t]] = az.loo(az.InferenceData(posterior=tiny, log_likelihood=ll), pointwise=True)
    PW[t] = loos[SHORT[t]].loo_i.values.copy()
    del ll, tiny
CMP = az.compare(loos, ic="loo")

fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.13,
                    subplot_titles=["elpd_loo · error bar = marginal SE, text = paired SE "
                                    "of the difference",
                                    "Pareto-k diagnostic (4,445 observations each)"])
order = [SHORT[t] for t in ("C", "B", "A")]
for nm in order:
    l = loos[nm]
    t = [k for k, v in SHORT.items() if v == nm][0]
    fig.add_trace(go.Scatter(x=[l.elpd_loo], y=[nm], mode="markers", name=nm,
                             legendgroup=t, marker=dict(color=FCOL[t], size=13, symbol="diamond",
                                                        line=dict(color="white", width=1)),
                             error_x=dict(type="data", array=[l.se], color=FCOL[t],
                                          thickness=1.5, width=6),
                             hovertemplate="%{y}: %{x:.1f}<extra></extra>"), row=1, col=1)
    d, se = CMP.loc[nm, "elpd_diff"], CMP.loc[nm, "dse"]
    fig.add_annotation(x=l.elpd_loo, y=nm, xshift=0, yshift=22, showarrow=False, row=1, col=1,
                       font=dict(size=10.5, color=C["dark"]),
                       text=f"{l.elpd_loo:.1f} ± {l.se:.1f}" +
                            ("" if d == 0 else f"   ·   −{d:.1f} ± {se:.1f} vs best"))
    k = l.pareto_k.values
    fig.add_trace(go.Bar(x=["k ≤ 0.5", "0.5 < k ≤ 0.7", "0.7 < k ≤ 1", "k > 1"],
                         y=[int((k <= .5).sum()), int(((k > .5) & (k <= .7)).sum()),
                            int(((k > .7) & (k <= 1)).sum()), int((k > 1).sum())],
                         name=nm, legendgroup=t, showlegend=False, marker_color=FCOL[t],
                         hovertemplate="%{x}: %{y}<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text="elpd_loo (± SE)", row=1, col=1)
fig.update_yaxes(categoryorder="array", categoryarray=order[::-1], tickfont=dict(size=10),
                 row=1, col=1)
fig.update_xaxes(title_text="Pareto shape k", row=1, col=2)
fig.update_yaxes(title_text="observations", row=1, col=2)
fig.update_layout(title="LOO-CV over the same 4,445 observations", height=470, width=1200,
                  bargap=0.25, legend=dict(orientation="h", y=-0.18), margin=dict(t=100, l=140))
show(fig, "12_loo")

print(CMP.to_string())
for x, y in [("C", "A"), ("B", "A"), ("C", "B")]:
    dd = PW[x] - PW[y]
    print(f"elpd({SHORT[x]}) − elpd({SHORT[y]}) = {dd.sum():+.1f} ± "
          f"{np.sqrt(len(dd)) * dd.std(ddof=1):.1f} (paired SE) = "
          f"{dd.sum() / (np.sqrt(len(dd)) * dd.std(ddof=1)):.1f} SE")
for t in ("A", "B", "C"):
    l = loos[SHORT[t]]
    print(f"{t}: p_loo {l.p_loo:.0f} on {len(PW[t])} obs · k>0.7 "
          f"{int((l.pareto_k.values > 0.7).sum())} · k>1 {int((l.pareto_k.values > 1).sum())} "
          f"· max k {l.pareto_k.values.max():.2f}")

/Users/yassineessifi/miniforge3/envs/pymc_env/lib/python3.11/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


/Users/yassineessifi/miniforge3/envs/pymc_env/lib/python3.11/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


/Users/yassineessifi/miniforge3/envs/pymc_env/lib/python3.11/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


                     rank     elpd_loo        p_loo  elpd_diff        weight         se        dse  warning scale
C · BM + noise gap      0  6808.732699  1622.759032   0.000000  9.885191e-01  76.757881   0.000000     True   log
B · BM + rate hier.     1  6735.105802  1581.499736  73.626897  1.148087e-02  72.219040  23.269660     True   log
A · no-BM               2  6708.996484  1584.604357  99.736215  5.148311e-20  71.502694  29.218577     True   log
elpd(C · BM + noise gap) − elpd(A · no-BM) = +99.7 ± 29.2 (paired SE) = 3.4 SE
elpd(B · BM + rate hier.) − elpd(A · no-BM) = +26.1 ± 13.8 (paired SE) = 1.9 SE
elpd(C · BM + noise gap) − elpd(B · BM + rate hier.) = +73.6 ± 23.3 (paired SE) = 3.2 SE
A: p_loo 1585 on 4445 obs · k>0.7 704 · k>1 62 · max k 1.41
B: p_loo 1581 on 4445 obs · k>0.7 703 · k>1 79 · max k 1.78
C: p_loo 1623 on 4445 obs · k>0.7 728 · k>1 93 · max k 2.88


### 13 · The elpd gain is a ceiling effect, not a new-benchmark effect.
GSM8K (**+41.9**) and FrontierMath v1 (**+31.8**) supply three quarters of C's +99.7; both are benchmarks where a ceiling caps the score. The 13 new benchmarks carry 38%, and **all** of that is the two FrontierMath v1 items with their fixed d = 0.57 / 0.60. GBAEval itself contributes **−0.5**.

In [14]:
dd = PW["C"] - PW["A"]
bn = np.array([BEN[i] for i in data.bench_idx])
T = pd.DataFrame({"bench": bn, "d": dd}).groupby("bench")["d"].agg(["sum", "size"])
T["grp"] = [G_IN if b in NEW_IN else G_OUT if b in NEW_OUT else G_OLD for b in T.index]
movers = T.reindex(T["sum"].abs().sort_values().index[-14:])
newb = T.loc[NEW].sort_values("sum")
newsum = T.loc[T.grp != G_OLD, "sum"].sum()

fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.34, 0.30], horizontal_spacing=0.16,
                    subplot_titles=["14 benchmarks that move most", "all 13 new benchmarks",
                                    "Where the +99.7 nats come from"])
seen = set()                       # a group is legended in the first panel that has it
for col, tab in [(1, movers), (2, newb)]:
    for g in (G_OLD, G_OUT, G_IN):
        sb = tab[tab.grp == g]
        if not len(sb):
            continue
        fig.add_trace(go.Bar(x=sb["sum"], y=list(sb.index), orientation="h", name=g,
                             legendgroup=g, showlegend=g not in seen, marker_color=GCOL[g],
                             hovertemplate="%{y}: %{x:+.1f} nats<extra></extra>"), row=1, col=col)
        seen.add(g)
    fig.add_vline(x=0, line=dict(color=C["dark"], width=1), row=1, col=col)
    fig.update_yaxes(categoryorder="array", categoryarray=list(tab.index),
                     tickfont=dict(size=9.5), row=1, col=col)
    fig.update_xaxes(title_text="Σ (elpd_C − elpd_A), nats", row=1, col=col)
fig.update_xaxes(range=[-24, 50], row=1, col=1)
fig.update_xaxes(range=[-6, 38], row=1, col=2)
for g in (G_IN, G_OUT, G_OLD):
    sb = T[T.grp == g]
    fig.add_trace(go.Bar(x=[g.replace(" / ", ",<br>")], y=[sb["sum"].sum()], marker_color=GCOL[g],
                         showlegend=False,
                         text=[f"{sb['sum'].sum():+.1f} nats<br>"
                               f"{100 * sb['sum'].sum() / dd.sum():+.0f}% · n={len(sb)}"],
                         textposition="outside", textfont=dict(size=11),
                         hovertemplate="%{y:+.1f} nats<extra></extra>"), row=1, col=3)
fig.add_hline(y=0, line=dict(color=C["dark"], width=1), row=1, col=3)
fig.update_yaxes(title_text="Σ (elpd_C − elpd_A), nats", range=[-22, 88], row=1, col=3)
fig.update_xaxes(tickangle=0, tickfont=dict(size=10), row=1, col=3)
fig.update_layout(title="Where C beats A: two ceiling-capped benchmarks, not the new arrivals",
                  height=520, width=1250, bargap=0.25,
                  legend=dict(orientation="h", y=-0.24), margin=dict(t=110, l=170, b=120))
show(fig, "13_loo_by_benchmark")

print(f"total C − A = {dd.sum():+.1f} nats · 13 new benchmarks {newsum:+.1f} "
      f"({100 * newsum / dd.sum():.0f}%), of which FrontierMath v1 pair "
      f"{T.loc[['FrontierMath v1', 'FrontierMath Tier 4 v1'], 'sum'].sum():+.1f}")
print("largest gains: " + " · ".join(f"{b} {T.loc[b, 'sum']:+.1f}" for b in T.index[-5:][::-1]))
print("largest losses: " + " · ".join(f"{b} {T.loc[b, 'sum']:+.1f}" for b in T.index[:5]))
print(T.loc[NEW, ["sum", "size"]].sort_values("sum", ascending=False).to_string())

total C − A = +99.7 nats · 13 new benchmarks +37.5 (38%), of which FrontierMath v1 pair +38.3
largest gains: WinoGrande +0.7 · WeirdML -1.8 · WMDP Chemistry -5.0 · WMDP Biology +13.4 · VisualToolBench +2.0
largest losses: APEX Agents +1.1 · ARC (AI2) -1.1 · ARC-AGI +3.2 · ARC-AGI-2 +11.8 · Adversarial NLI +0.0
                              sum  size
bench                                  
FrontierMath v1         31.770257   101
FrontierMath Tier 4 v1   6.530802    72
DeepSWE                  3.327640    37
EBR-bench                0.797355    17
SpatialViz-Bench         0.112338     8
BlueprintBench 2         0.058362    17
AlgoTune                 0.010547    18
MindCube                 0.003109     5
GDP.pdf                 -0.301100     9
ProofBench              -0.429366    44
GBAEval                 -0.493125    14
GDPval                  -1.334660    11
Surface Evolver Bench   -2.526620    17


### Verdict
Brownian lineage steps improve prediction (**+99.7 ± 29.2** nats, C over A) and cost nothing in convergence (**18 / 12,000** divergences), but they do not touch the basin problem: all three fits split 2 ways on the same question.
That question is GBAEval — easy-knowledge loading **2.4** in one basin, **0.2** in the other — a 14-model benchmark with **zero** model overlap with all 7 classic commonsense items and a near-separating score distribution.
The two extra structures are empty: 36 vendor rates inside **9–20%** of one vendor's own interval, and a noise gap above 0.10 on **3 of 98** benchmarks.
The elpd gain is a **ceiling** effect (GSM8K +41.9, FrontierMath v1 +31.8), not a new-benchmark effect; PSIS is unreliable at p_loo ≈ 1,600, so treat the ranking as ordinal.
The lever is coverage — co-measure GBAEval against classic items — not another lineage parameterisation.